In [0]:
import json 
from pyspark.sql.functions import *

kafka_connection_json=dbutils.secrets.get(scope="frauddetect-scope",key="kafka_connection_details")
kafka_config=json.loads(kafka_connection_json)
bootstrap_server=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic_name =kafka_config['topic']

## java authentication protocol
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

# sample_batch = (spark.read.format('kafka')
#                       .option("subscribe", topic_name)
#                       .option("kafka.bootstrap.servers", bootstrap_server)
#                         .option("kafka.sasl.mechanism", "PLAIN")
#                       .option("kafka.security.protocol", "SASL_SSL")
#                       .option("kafka.sasl.jaas.config", jaas_config)
#                       .option("startingOffsets","earliest")
#                       .load()) 


# parsed_batch=sample_batch.select(
# col("key").cast("string"),
# col("value").cast("string"),
# col("topic"),
# col("partition"),
# col("offset"),
# col("timestamp"),
# col("timestampType")
# )
# display(parsed_batch)

# parsed_batch.write.saveAsTable("fraud_detect.bronze.transactions_batch_test")

# sample_batch.count() 

In [0]:
from pyspark.sql.functions import * 
streaming_df=(spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_server)
              .option("subscribe",topic_name)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest")
        .load()) 



parsed_streaming_df=streaming_df.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

streaming_query=(parsed_streaming_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/fraud_detect/source/transactions/checkpoint/")
.trigger(availableNow=True)
.toTable("fraud_detect.bronze.transactions_streaming_test")
)
print("Query Started with query id: ",streaming_query.id)

stream_df = spark.sql(f""" 
          select key,value,topic,offset from fraud_detect.bronze.transactions_streaming_test
          """)
display(stream_df)
